# Grad-CAM for one-dimensional biological trajectories

The goal is to identify temporal regions that support a classifier's prediction for control, high TGF-beta or high GDF11 trajectories.

## 1. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from tensorflow import keras

from deeplearning_examples.timecourse import load_smad_classification_data
from tensorflow_gradcam.gradcam import compute_gradcam, plot_gradcam
from tensorflow_gradcam.model import build_model
from tensorflow_gradcam.training import TrainingConfig, train

## 2. Data and held-out split

Grad-CAM is evaluated on cells that are not used for fitting. This avoids explaining memorized training examples.

In [ ]:
dataset = load_smad_classification_data()
x_train, x_test, y_train_int, y_test_int = train_test_split(
    dataset.channels_last,
    dataset.targets,
    test_size=0.25,
    random_state=42,
    stratify=dataset.targets,
)
y_train = keras.utils.to_categorical(y_train_int, len(dataset.class_names))
y_test = keras.utils.to_categorical(y_test_int, len(dataset.class_names))

## 3. Build and train the classifier

The final convolutional layer is named explicitly so the attribution code can address it without relying on a fragile numeric layer index.

In [ ]:
model = build_model(input_length=x_train.shape[1], num_classes=len(dataset.class_names))
model.summary()
history = train(model, x_train, y_train, config=TrainingConfig(epochs=200, patience=20))

### Optimization diagnostics

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].plot(history.history["loss"], label="train")
axes[0].plot(history.history["val_loss"], label="validation")
axes[0].set_title("Loss")
axes[1].plot(history.history["categorical_accuracy"], label="train")
axes[1].plot(history.history["val_categorical_accuracy"], label="validation")
axes[1].set_title("Accuracy")
for axis in axes: axis.legend()
figure.tight_layout()

## 4. Held-out performance

Attribution is only meaningful if the underlying prediction is sufficiently reliable.

In [ ]:
probabilities = model.predict(x_test, verbose=0)
predictions = probabilities.argmax(1)
print(classification_report(y_test_int, predictions, target_names=dataset.class_names))

## 5. Temporal Grad-CAM

The heatmap weights the last convolutional feature maps by the gradient of one class score. It is interpolated back to the original time axis for visualization.

In [ ]:
example_index = int(np.flatnonzero(predictions == y_test_int)[0])
target_class = int(predictions[example_index])
heatmap = compute_gradcam(model, x_test[example_index], class_index=target_class)

figure, axis = plt.subplots(figsize=(10, 3.5))
plot_gradcam(axis, dataset.times, x_test[example_index], heatmap)
axis.set_title(
    f"true: {dataset.class_names[y_test_int[example_index]]}; "
    f"predicted: {dataset.class_names[target_class]}"
)
figure.tight_layout()

## 6. Compare explanations across classes

Single examples are unstable evidence. Sampling correctly classified cells from every class helps reveal whether the same temporal region is used consistently.

In [ ]:
figure, axes = plt.subplots(len(dataset.class_names), 3, figsize=(12, 8), sharex=True, sharey=True)
for class_index, class_name in enumerate(dataset.class_names):
    candidates = np.flatnonzero((y_test_int == class_index) & (predictions == class_index))[:3]
    for axis, sample_index in zip(axes[class_index], candidates):
        heatmap = compute_gradcam(model, x_test[sample_index], class_index=class_index)
        plot_gradcam(axis, dataset.times, x_test[sample_index], heatmap)
        axis.set_title(class_name)
figure.tight_layout()

## 7. Interpretation limits

Grad-CAM is a model attribution, not a direct measurement of biological causality. Robust conclusions should test sensitivity across random seeds, target layers and perturbations of the highlighted interval.